# 04 - Diagnostics and Figures  (Stage 4)

**Purpose.** Produce the pathway map, seasonal & interannual fractions, transit
time distributions, centroid map, and the pathway summary table for the chosen
`k`.

**Input.** `data/labeled_trajectories.parquet` (Stage 3), the chosen k-means
model, and the original Zarr stores (for full trajectory paths / transit times).
**Output.** PNGs in `figures/` and a printed summary table.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)

**Raw clusters vs. merged groups.** Set `LABEL_COL = "cluster_group"` to make
every figure at the merged-pathway level (requires a `GROUP_MAP` in `config`),
or `"cluster_label"` for the raw k-means clusters. Everything below keys off
`LABEL_COL`, so the figures, fractions and table all follow your choice.

In [ ]:
BEST_K = 20                    # <-- must match notebook 03 (papermill: -p BEST_K <k>)
LABEL_COL = "cluster_group"    # "cluster_group" (merged) or "cluster_label" (raw)

In [ ]:
import pickle, matplotlib.pyplot as plt
lab = pd.read_parquet(C.LABELED_FILE)
with open(C.MODELS_DIR / f"kmeans_k{BEST_K}.pkl", "rb") as f:
    km = pickle.load(f)
labelled = lab[lab[LABEL_COL] >= 0].copy()
n_lab = int(labelled[LABEL_COL].max()) + 1
cmap = plt.get_cmap("tab20", max(n_lab, 3))

# Display centroids in real degrees, one row per label (lat50, lon50, lat100, lon100).
raw_cent = P.centroids_to_degrees(km.cluster_centers_)
if LABEL_COL == "cluster_group" and C.GROUP_MAP:
    _, raw2grp = P.apply_group_map(np.arange(BEST_K), C.GROUP_MAP, BEST_K)
    sizes = lab.loc[lab.cluster_label >= 0, "cluster_label"].value_counts()
    # Merge in the *native* feature space (xyz or cosine), size-weighted, then
    # reproject -> a proper spherical group centroid in xyz mode (not a naive
    # lat/lon average).
    dim = km.cluster_centers_.shape[1]
    grp_centers = np.zeros((n_lab, dim)); wsum = np.zeros(n_lab)
    for c in range(BEST_K):
        g = raw2grp[c]; w = float(sizes.get(c, 0))
        grp_centers[g] += km.cluster_centers_[c] * w; wsum[g] += w
    grp_centers /= np.where(wsum[:, None] == 0, 1, wsum[:, None])
    disp_cent = P.centroids_to_degrees(grp_centers)
else:
    disp_cent = raw_cent
LABEL_NAMES = C.GROUP_NAMES if LABEL_COL == "cluster_group" else {}
print(f"labelled trajectories: {len(labelled):,} | {LABEL_COL}: {n_lab} labels")

## Figure 1 - Pathway map

Plot up to ~5,000 *full* trajectories per cluster, coloured by label. Full paths
are streamed from the Zarr stores so we never hold everything in memory. We pick
a random set of `trajectory_id`s per cluster, group them by store, and read only
those rows.

In [ ]:
import cartopy.crs as ccrs, cartopy.feature as cfeature
rng = np.random.default_rng(C.RANDOM_STATE)

# choose <=5000 trajectory_ids per label (raw cluster or merged group)
pick = (labelled.groupby(LABEL_COL, group_keys=False)
        .apply(lambda d: d.sample(min(5000, len(d)), random_state=C.RANDOM_STATE)))
pick = pick[["trajectory_id", LABEL_COL]].rename(columns={LABEL_COL: "lab"}).copy()
pick["store_index"] = pick.trajectory_id // C.TRAJ_PER_STORE
pick["local"] = pick.trajectory_id % C.TRAJ_PER_STORE
stores = C.list_stores()

fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([C.DOMAIN["lon_min"], C.DOMAIN["lon_max"],
               C.DOMAIN["lat_min"], C.DOMAIN["lat_max"]], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)

import xarray as xr
for si, grp in pick.groupby("store_index"):
    ds = xr.open_zarr(stores[si])
    loc = grp.local.to_numpy()
    lons = ds.lon.values[loc]; lats = ds.lat.values[loc]
    cl = grp.lab.to_numpy()
    for j in range(len(loc)):
        m = np.isfinite(lons[j]) & np.isfinite(lats[j])   # drop NaN tails (deleted particles)
        if not m.any():
            continue
        ax.plot(lons[j][m], lats[j][m], color=cmap(cl[j]), lw=.2, alpha=.3,
                transform=ccrs.PlateCarree())
ax.set_title(f"Plume pathways by {LABEL_COL} (k={BEST_K}, {n_lab} labels)")
fig.savefig(C.FIG_DIR / f"pathway_map_k{BEST_K}_{LABEL_COL}.png", dpi=140, bbox_inches="tight")
plt.show()

## Figure 2 - Seasonal pathway fractions (stacked bars by release month)

In [ ]:
def label_name(c):
    return LABEL_NAMES.get(c, str(c))

def fraction_table(group_col):
    g = labelled.groupby([group_col, LABEL_COL]).size().unstack(fill_value=0)
    return g.div(g.sum(axis=1), axis=0) * 100

seas = fraction_table("release_month").reindex(range(1, 13))
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(seas))
for c in range(n_lab):
    vals = seas.get(c, pd.Series(0, index=seas.index)).to_numpy()
    ax.bar(seas.index, vals, bottom=bottom, color=cmap(c), label=label_name(c))
    bottom += vals
ax.set_xlabel("release month"); ax.set_ylabel("% of particles")
ax.set_title(f"Seasonal pathway fractions ({LABEL_COL})")
ax.legend(ncol=2, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.savefig(C.FIG_DIR / f"seasonal_fractions_k{BEST_K}_{LABEL_COL}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 3 - Interannual pathway fractions (stacked area by release year)

In [ ]:
yr = fraction_table("release_year").sort_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.stackplot(yr.index, *[yr.get(c, pd.Series(0, index=yr.index)).to_numpy()
                         for c in range(n_lab)],
             colors=[cmap(c) for c in range(n_lab)], labels=[label_name(c) for c in range(n_lab)])
ax.set_xlabel("release year"); ax.set_ylabel("% of particles"); ax.set_ylim(0, 100)
ax.set_title(f"Interannual pathway fractions ({LABEL_COL})")
ax.legend(ncol=2, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.savefig(C.FIG_DIR / f"interannual_fractions_k{BEST_K}_{LABEL_COL}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 4 - Transit-time distributions per cluster

Time (days since release) to first reach 10°N, computed from the Zarr stores via
`pipeline.first_crossing_days`. We reuse the per-cluster `pick` sample from
Figure 1 to keep the I/O bounded.

In [ ]:
transit = {}   # trajectory_id -> days to 10N
for si, grp in pick.groupby("store_index"):
    days = P.first_crossing_days(stores[si], lat_thresh=10.0)
    for tid, loc in zip(grp.trajectory_id, grp.local):
        transit[tid] = days[loc]
pick["t10N"] = pick.trajectory_id.map(transit)

ncol = 5; nrow = int(np.ceil(n_lab / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 2.4*nrow), squeeze=False)
for c in range(n_lab):
    ax = axes[c // ncol][c % ncol]
    d = pick.loc[pick.lab == c, "t10N"].dropna()
    ax.hist(d, bins=30, color=cmap(c))
    ax.set_title(f"{label_name(c)} (n={len(d)})", fontsize=8)
    ax.set_xlabel("days to 10N", fontsize=7)
for j in range(n_lab, nrow*ncol):
    axes[j // ncol][j % ncol].axis("off")
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"transit_time_histograms_k{BEST_K}_{LABEL_COL}.png", dpi=120)
plt.show()

## Figure 5 - Centroid positions (day-50 circles, day-100 stars)

In [ ]:
cent = disp_cent     # lat50, lon50, lat100, lon100 (per label, real degrees)
pct = labelled[LABEL_COL].value_counts(normalize=True).sort_index() * 100
fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([C.DOMAIN["lon_min"], C.DOMAIN["lon_max"],
               C.DOMAIN["lat_min"], C.DOMAIN["lat_max"]], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)
for c in range(n_lab):
    ax.scatter(cent[c, 1], cent[c, 0], color=cmap(c), s=120, marker="o",
               edgecolor="k", transform=ccrs.PlateCarree())
    ax.scatter(cent[c, 3], cent[c, 2], color=cmap(c), s=200, marker="*",
               edgecolor="k", transform=ccrs.PlateCarree())
    ax.text(cent[c, 3], cent[c, 2], f" {label_name(c)} ({pct.get(c,0):.0f}%)", fontsize=8,
            transform=ccrs.PlateCarree())
ax.set_title(f"Centroids ({LABEL_COL}, k={BEST_K})  o=day50  *=day100")
fig.savefig(C.FIG_DIR / f"centroids_k{BEST_K}_{LABEL_COL}.png", dpi=140, bbox_inches="tight")
plt.show()

## Table 1 - Pathway summary

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
rows = []
for c in range(n_lab):
    sel = labelled[labelled[LABEL_COL] == c]
    peak = month_names[int(sel.release_month.mode().iloc[0]) - 1] if len(sel) else "-"
    mt = pick.loc[pick.lab == c, "t10N"].dropna()
    rows.append(dict(
        label=c,
        name=LABEL_NAMES.get(c, ""),
        pct=round(len(sel) / len(labelled) * 100, 1),
        peak_month=peak,
        mean_transit_10N=round(float(mt.mean()), 1) if len(mt) else np.nan,
        mean_lat_100=round(float(sel.lat_100.mean()), 2),
        mean_lon_100=round(float(sel.lon_100.mean()), 2),
    ))
summary = pd.DataFrame(rows)
summary.to_csv(C.DATA_DIR / f"pathway_summary_k{BEST_K}_{LABEL_COL}.csv", index=False)
print(summary.to_string(index=False))